In [1]:
# !pip install thefuzz
# !pip install mosestokenizer

In [2]:
import subprocess
from pathlib import Path
import json

import random
import pandas as pd
from sklearn.model_selection import train_test_split

# from tqdm.auto import tqdm
# tqdm.pandas()

from thefuzz import fuzz
# from sacremoses import MosesTokenizer, MosesDetokenizer
# from nltk.tokenize import sent_tokenize
from mosestokenizer import *

seed = 42
random.seed(seed)

level_map = {'adv': 0, 'int': 1, 'ele': 2}

In [3]:
# set pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 10)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [4]:
data_dir = Path("../resources/data/en/OneStopEnglishCorpus/")
output_dir = Path("../resources/data/en/OneStopEnglishCorpus/splits/")
output_dir.mkdir(parents=True, exist_ok=True)

In [5]:
def generate_dataframe_from_directory(dir_path):
    file_list = {'article': [file.name for file in sorted(Path(dir_path).iterdir()) if file.suffix == '.txt']}
    return pd.DataFrame(file_list)

def split_and_concat_data(df):
    train, test = train_test_split(df, test_size=0.2, random_state=42)
    train, valid = train_test_split(train, test_size=0.05, random_state=42)  # 0.25 x 0.8 = 0.2
    
    train['split'] = 'train'
    test['split'] = 'test'
    valid['split'] = 'valid'

    # Concatenate the splits back into a single dataframe
    df = pd.concat([train, test, valid], ignore_index=True)
    
    return df

# Example usage:
dfs = []
for level in ["Adv", "Int", "Ele"]:
    df = generate_dataframe_from_directory(data_dir / f"Texts-SeparatedByReadingLevel/{level}-Txt")
    # remove level from file 
    df['article'] = df['article'].apply(lambda x: x.replace(f"-{level.lower()}.txt", ""))
    # df.columns = [level.lower()]
    dfs.append(df)

# ensure rows match across all dataframes
meta_df = pd.merge(dfs[0], dfs[1], on='article', how='outer')
meta_df = pd.merge(meta_df, dfs[2], on='article', how='outer')
meta_df = split_and_concat_data(meta_df)

print(meta_df['split'].value_counts())

meta_df.head()


train    143
test      38
valid      8
Name: split, dtype: int64


,article,split
0,WNL Bangladeshi organization,train
1,WNL Shakespeare,train
2,WNL Pacific Islanders,train
3,NSA 2,train
4,Coal to challenge oil,train


In [6]:
# load aligned sentences for ADV-ELE.txt
def load_aligned_sentences(tgt_level):
    infile = data_dir / "Sentence-Aligned" / f"ADV-{tgt_level}.txt"
    current_pair = []
    c = 0
    with open(infile, 'r', encoding='utf8') as f:
        for line in f:
            line = line.strip()
            if line == '*******' and len(current_pair) == 2:
                yield {f"src": current_pair[0], f"tgt": current_pair[1]}
                c += 1
                current_pair = []
            else:
                current_pair.append(line.strip())

        # last pair
        if len(current_pair) == 2:
            yield {f"src": current_pair[0], f"tgt": current_pair[1]}
            c += 1

    print(f"Loaded {c} pairs from {infile}")

adv_int = pd.DataFrame(list(load_aligned_sentences("INT")))
adv_ele = pd.DataFrame(list(load_aligned_sentences("ELE")))
adv_int.head()

Loaded 2154 pairs from ../resources/data/en/OneStopEnglishCorpus/Sentence-Aligned/ADV-INT.txt
Loaded 2166 pairs from ../resources/data/en/OneStopEnglishCorpus/Sentence-Aligned/ADV-ELE.txt


,src,tgt
0,Brazil and Peru have lodged objections to a bid made by the US e-commerce giant for a prime new piece of cyberspace: .amazon.,Brazil and Peru have made objections to a bid made by the huge US e-commerce company for a prime new piece of cyberspace: .amazon.
1,"Until now, the differences between commercial, governmental and other types of identity were easily distinguished in every internet address by .com, .gov and 20 other categories.","Until now, the differences between commercial, governmental and other types of identity were easy to see in every internet address by the use of .com, .gov and 20 other categories."
2,But these categories or generic top-level domains (gTLDs) as they are technically known are about to undergo the biggest expansion since the start of the worldwide web.,But these categories or generic top-level domains (gTLDs) as they are technically known are about to see the biggest expansion since the start of the worldwide web.
3,"Amazon has applied for dozens of new domains, including .shop, .song, .book and .kindle.","Amazon has applied for many new domains, including .shop, .song, .book and .kindle."
4,"Allowing private companies to register geographical names as gTLDs to reinforce their brand strategy or to profit from the meaning of these names does not serve, in our view, the public interest, the Brazilian Ministry of Science and Technology said.","Allowing private companies to register geographical names as gTLDs to strengthen their brand or to profit from the meaning of these names is not, in our view, in the public interest, the Brazilian Ministry of Science and Technology said."


In [7]:
# tokenize texts
from mosestokenizer import MosesSentenceSplitter

def sent_tokenize_text(text):
    with MosesSentenceSplitter('en') as splitsents:
        return splitsents(text)

output_dir = Path("../resources/data/en/OneStopEnglishCorpus/Texts-SeparatedByReadingLevel-SentTok/")
output_dir.mkdir(parents=True, exist_ok=True)

for level in ["Adv", "Int", "Ele"]:
    for file in (data_dir / "Texts-SeparatedByReadingLevel" / f"{level}-Txt").glob("*txt"):
        outfile = output_dir / f"{level}-Txt" / file.name
        outfile.parent.mkdir(parents=True, exist_ok=True)
        with open(file, 'r', encoding='utf-8-sig') as f: # utf-8-sig to handle BOM
            # for some reason, intermediate texts start with 'Intermediate'
            text = [line.strip() for line in f if line.strip() and line.strip() != 'Intermediate']
            sents = sent_tokenize_text(text)
            with open(outfile, 'w', encoding='utf-8') as out:
                out.write("\n".join(sents))
                # print(f"Written {len(sents)} sentences to {outfile}")

In [8]:
# # for each sentence pair, find the corresponding text file name by searching text files in the directory
def find_article_containing_text(text, dir_path):
    dir_path = Path(dir_path).resolve()
    c = 0

    for file in Path(dir_path).glob('*.txt'):
        with open(file, 'r', encoding='utf8') as f:
            c += 1
            if text in f.read():
                print(f"Found text in {file.name} after checking {c} files for {text}")
                return file.name
    print(f"Failed to find text in files (Checked {c} files for {text}")
    return None

# print(find_file_containing_text(adv_int['src'][0], data_dir / "Texts-SeparatedByReadingLevel/Adv-Txt"))

def fuzzy_find_article_containing_text(aligned_pair, tgt_level, dir_path, threshold=90):
    dir_path = Path(dir_path).resolve()
    c = 0

    source_level_texts = dir_path / "Adv-Txt"
    target_level_texts = dir_path / f"{tgt_level}-Txt"

    for file in Path(source_level_texts).glob('*.txt'):
        with open(file, 'r', encoding='utf8') as f:
            c += 1
            if fuzz.partial_ratio(aligned_pair[0], f.read()) > threshold:
                # print(f"Found text in {file.name} after checking {c} files for {text}")
                return file.name[:-8]

    for file in Path(target_level_texts).glob('*.txt'):
        with open(file, 'r', encoding='utf8') as f:
            c += 1
            if fuzz.partial_ratio(aligned_pair[1], f.read()) > threshold:
                # print(f"Found text in {file.name} after checking {c} files for {text}")
                return file.name[:-8]

    print(f"[!] Failed to find text in files ({aligned_pair[0]})")
    return None

def match_sentences_to_articles(df, tgt_level, dir_path):
    df['article'] = df.apply(lambda x: fuzzy_find_article_containing_text((x['src'], x['tgt']), tgt_level, dir_path) , axis=1)
    return df

# print(fuzzy_find_file_containing_text((adv_int['src'][0], adv_int['tgt'][0]), 'Int', data_dir / "Texts-SeparatedByReadingLevel-SentTok"))
# adv_int['article'] = adv_int.apply(lambda x: fuzzy_find_file_containing_text((x['src'], x['tgt']), 'Int', data_dir / "Texts-SeparatedByReadingLevel-SentTok") , axis=1)

adv_int = match_sentences_to_articles(adv_int, 'Int', data_dir / "Texts-SeparatedByReadingLevel-SentTok")
adv_ele = match_sentences_to_articles(adv_ele, 'Ele', data_dir / "Texts-SeparatedByReadingLevel-SentTok")

[!] Failed to find text in files (Faehrmann said the protests had shown Australians wanted sharks protected: Whats amazing is so many people in Australia love sharks.)
[!] Failed to find text in files (Researchers at the University of Western Australia say the recent spate of shark attacks in the state may have more to do with the state having the fastest-growing population in Australia, rather than a rising number of sharks.)
[!] Failed to find text in files (Faehrmann said the protests had shown Australians wanted sharks protected: Whats amazing is so many people in Australia love sharks.)
[!] Failed to find text in files (Researchers at the University of Western Australia say the recent spate of shark attacks in the state may have more to do with the state having the fastest-growing population in Australia, rather than a rising number of sharks.)
[!] Failed to find text in files (Faehrmann said the protests had shown Australians wanted sharks protected: Whats amazing is so many peop

In [9]:
adv_ele.head()

,src,tgt,article
0,"The Seattle-based company has applied for its brand to be a top-level domain name (currently .com), but the South American governments argue this would prevent the use of this internet address for environmental protection, the promotion of indigenous rights and other public interest uses.","Amazon has asked for its company name to be a top-level domain name (currently .com), but the South American governments say this would stop the use of this internet address for environmental protection, indigenous rights and other public interest uses.",Amazon
1,"Until now, the differences between commercial, governmental and other types of identity were easily distinguished in every internet address by .com, .gov and 20 other categories.","Until now, the differences between commercial, governmental and other types of identity were easy to see in every internet address by the use of .com, .gov and 20 other categories.",Amazon
2,"Amazon has applied for dozens of new domains, including .shop, .song, .book and .kindle.","Amazon has applied for many new domains, including .shop, .song, .book and .kindle.",Amazon
3,"Allowing private companies to register geographical names as gTLDs to reinforce their brand strategy or to profit from the meaning of these names does not serve, in our view, the public interest, the Brazilian Ministry of Science and Technology said.","Allowing private companies to register geographical names as gTLDs to profit from the meaning of these names is not, in our view, in the public interest, the Brazilian Ministry of Science and Technology said.",Amazon
4,"Brazil said its views were endorsed last month by other members of the Amazon Cooperation Treaty (Bolivia, Colombia, Ecuador, Guyana, Suriname and Venezuela).","Brazil said other members of the Amazon Cooperation Treaty support its views (Bolivia, Colombia, Ecuador, Guyana, Suriname and Venezuela).",Amazon


In [10]:
adv_int.head()

,src,tgt,article
0,Brazil and Peru have lodged objections to a bid made by the US e-commerce giant for a prime new piece of cyberspace: .amazon.,Brazil and Peru have made objections to a bid made by the huge US e-commerce company for a prime new piece of cyberspace: .amazon.,Amazon
1,"Until now, the differences between commercial, governmental and other types of identity were easily distinguished in every internet address by .com, .gov and 20 other categories.","Until now, the differences between commercial, governmental and other types of identity were easy to see in every internet address by the use of .com, .gov and 20 other categories.",Amazon
2,But these categories or generic top-level domains (gTLDs) as they are technically known are about to undergo the biggest expansion since the start of the worldwide web.,But these categories or generic top-level domains (gTLDs) as they are technically known are about to see the biggest expansion since the start of the worldwide web.,Amazon
3,"Amazon has applied for dozens of new domains, including .shop, .song, .book and .kindle.","Amazon has applied for many new domains, including .shop, .song, .book and .kindle.",Amazon
4,"Allowing private companies to register geographical names as gTLDs to reinforce their brand strategy or to profit from the meaning of these names does not serve, in our view, the public interest, the Brazilian Ministry of Science and Technology said.","Allowing private companies to register geographical names as gTLDs to strengthen their brand or to profit from the meaning of these names is not, in our view, in the public interest, the Brazilian Ministry of Science and Technology said.",Amazon


In [11]:
def get_split_info(df, meta_df=meta_df):
    # drop rows where article is None
    l1 = len(df)
    sdf = df[df['article'].isna()]
    if not sdf.empty:
        for i, row in sdf.iterrows():
            print(f"Missing article for {row['src']}")
        print(f"Dropping {len(sdf)} rows with missing article")
        df = df.dropna(subset=['article'])

    l2 = len(df)
    # df = df.dropna(subset=['article'])
    # l2 = len(df)
    # print(f"Dropped {l2 - l1} rows with missing article")

    # get split info from meta_df
    df = df.merge(meta_df, on='article', how='left')
    l3 = len(df)
    print(f"Dropped {l3 - l2} rows with missing split info")    

    # drop any rows containing none values
    sdf = df[df.isna().any(axis=1)]
    if not sdf.empty:
        print(f"Dropping {len(sdf)} rows with missing values")
        for i, row in sdf.iterrows():
            print(f"Missing values for {row['src']} in {row['article']}")
        df = df.dropna()

    print(df['split'].value_counts())

    return df

adv_int_splits = get_split_info(adv_int)
adv_ele_splits = get_split_info(adv_ele)


Dropped 0 rows with missing split info
train    1601
test      447
valid     106
Name: split, dtype: int64
Missing article for Faehrmann said the protests had shown Australians wanted sharks protected: Whats amazing is so many people in Australia love sharks.
Missing article for Researchers at the University of Western Australia say the recent spate of shark attacks in the state may have more to do with the state having the fastest-growing population in Australia, rather than a rising number of sharks.
Missing article for Faehrmann said the protests had shown Australians wanted sharks protected: Whats amazing is so many people in Australia love sharks.
Missing article for Researchers at the University of Western Australia say the recent spate of shark attacks in the state may have more to do with the state having the fastest-growing population in Australia, rather than a rising number of sharks.
Missing article for Faehrmann said the protests had shown Australians wanted sharks protect

In [12]:
# write aligned sentences to split files
def write_aligned_sentences_to_files(df, tgt_level, out_dir):
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    
    for split in ['train', 'valid', 'test']:
        split_df = df[df['split'] == split]
        with open(Path(out_dir) / f"onestopenglish_l0_l{level_map[tgt_level]}_{split}.tsv", 'w', encoding='utf8') as f:
            for i, row in split_df.iterrows():
                try:
                    if row['src'].strip() == '' or row['tgt'].strip() == '':
                        continue
                    else:
                        f.write(f"{row['src']}\t{row['tgt']}\n")
                except Exception as e:
                    print(f"Error writing {row['src']} - {row['tgt']} to file: {e}")
    return

write_aligned_sentences_to_files(adv_int_splits, 'int', out_dir='../resources/data/en/aligned/')
write_aligned_sentences_to_files(adv_ele_splits, 'ele', out_dir='../resources/data/en/aligned/')

In [13]:
# gather paragraphs from original text files for each split

for level in ['Adv', 'Int', 'Ele']:
    level_dir = data_dir / f"Texts-SeparatedByReadingLevel/{level}-Txt"
    for split in ['train', 'valid', 'test']:
        splif_meta_df = meta_df[meta_df['split'] == split]

        out_file = Path('../resources/data/en/OneStopEnglishCorpus') / 'splits' / f'{split}_{level_map[level.lower()]}.txt'
        out_file.parent.mkdir(parents=True, exist_ok=True)
        c = 0
        # for each article in the split, fetch the text from the original file and write to the split file
        with open(out_file, 'w', encoding='utf8') as outf:
            for i, row in splif_meta_df.iterrows():
                with open(level_dir / f"{row['article']}-{level.lower()}.txt", 'r', encoding='utf8') as inf:
                    lines = [line.strip() for line in inf if line.strip() and line.strip() != 'Intermediate']
                    for line in lines:
                        outf.write(line + '\n')
                        c += 1
        print(f"Written {c} lines to {out_file}")


Written 1985 lines to ../resources/data/en/OneStopEnglishCorpus/splits/train_0.txt
Written 108 lines to ../resources/data/en/OneStopEnglishCorpus/splits/valid_0.txt
Written 557 lines to ../resources/data/en/OneStopEnglishCorpus/splits/test_0.txt
Written 1801 lines to ../resources/data/en/OneStopEnglishCorpus/splits/train_1.txt
Written 102 lines to ../resources/data/en/OneStopEnglishCorpus/splits/valid_1.txt
Written 503 lines to ../resources/data/en/OneStopEnglishCorpus/splits/test_1.txt
Written 1618 lines to ../resources/data/en/OneStopEnglishCorpus/splits/train_2.txt
Written 90 lines to ../resources/data/en/OneStopEnglishCorpus/splits/valid_2.txt
Written 442 lines to ../resources/data/en/OneStopEnglishCorpus/splits/test_2.txt


In [ ]:
# prepare data for supervised training while we're at it!

indir = Path('../resources/data/en/aligned/')
outdir = Path('../resources/supervised/onestopenglish/data')

for split in ['train', 'valid', 'test']:
    outfile = outdir / f'{split}.json'
    with open(outfile, 'w', encoding='utf8') as outf:
        for tgt_level in [1, 2]:
            level_tag = f'<L{tgt_level}> '
            with open(indir / f'onestopenglish_l0_l{tgt_level}_{split}.tsv', 'r', encoding='utf8') as inf:
                for i, line in enumerate(inf):
                    src, tgt = line.strip().split('\t')
                    d = {"complex": level_tag+src, "simple": tgt}
                    outf.write(json.dumps(d, ensure_ascii=False) + '\n')
        print(f'Wrote {i} lines to {outfile}')

# shuffle the train.json file
with open(outdir / 'train.json', 'r', encoding='utf8') as f:
    lines = f.readlines()
    random.shuffle(lines)
with open(outdir / 'train.json', 'w', encoding='utf8') as f:
    f.writelines(lines)

In [3]:
# prepare data for muss as well

indir = Path('../resources/data/en/aligned/')
for level in [1, 2]:
    outdir = Path(f'../resources/muss/resources/datasets/onestopenglish_l{level}')
    outdir.mkdir(parents=True, exist_ok=True)
    for split in ['train', 'valid', 'test']:
        infile = indir / f'onestopenglish_l0_l{level}_{split}.tsv'
        src_outfile = outdir / f'{split}.complex'
        tgt_outfile = outdir / f'{split}.simple'
        with open(src_outfile, 'w', encoding='utf8') as srcf, open(tgt_outfile, 'w', encoding='utf8') as tgtf:
            with open(infile, 'r', encoding='utf8') as inf:
                for i, line in enumerate(inf):
                    src, tgt = line.strip().split('\t')
                    srcf.write(src + '\n')
                    tgtf.write(tgt + '\n')
        print(f'Wrote {i} lines to {src_outfile} and {tgt_outfile}')


Wrote 1600 lines to ../resources/muss/resources/datasets/onestopenglish_l1/train.complex and ../resources/muss/resources/datasets/onestopenglish_l1/train.simple
Wrote 105 lines to ../resources/muss/resources/datasets/onestopenglish_l1/valid.complex and ../resources/muss/resources/datasets/onestopenglish_l1/valid.simple
Wrote 446 lines to ../resources/muss/resources/datasets/onestopenglish_l1/test.complex and ../resources/muss/resources/datasets/onestopenglish_l1/test.simple
Wrote 1621 lines to ../resources/muss/resources/datasets/onestopenglish_l2/train.complex and ../resources/muss/resources/datasets/onestopenglish_l2/train.simple
Wrote 75 lines to ../resources/muss/resources/datasets/onestopenglish_l2/valid.complex and ../resources/muss/resources/datasets/onestopenglish_l2/valid.simple
Wrote 461 lines to ../resources/muss/resources/datasets/onestopenglish_l2/test.complex and ../resources/muss/resources/datasets/onestopenglish_l2/test.simple
